# Episode 2 — JIT, Tracing, and the Jaxpr

**Instructor notebook** · run top-to-bottom before recording.

`jax.jit` traces your function once into a **jaxpr**, lowers to XLA, and caches the executable.

| | |
|---|---|
| **Chapter** | 1.2 · Part I — Pure JAX |
| **Prereq** | Episode 1 |
| **Next** | Episode 3 — automatic differentiation |

**JAX docs:** [Just-in-time compilation](https://docs.jax.dev/en/latest/jit-compilation.html) · [Tracing](https://docs.jax.dev/en/latest/tracing.html) · [jaxpr](https://docs.jax.dev/en/latest/jaxpr.html) · [Control flow with JIT](https://docs.jax.dev/en/latest/control-flow.html) · [`jax.make_jaxpr`](https://docs.jax.dev/en/latest/_autosummary/jax.make_jaxpr.html) · [`jax.jit`](https://docs.jax.dev/en/latest/_autosummary/jax.jit.html)


In [ ]:
from functools import partial

import jax
import jax.numpy as jnp
import jax.random as jr
from jax import make_jaxpr

## How JAX transformations work

JAX reduces your function to a sequence of **primitives** — one fundamental op each. `make_jaxpr` shows that IR. Transformations only understand **side-effect-free** (pure) code: Python side effects are invisible to the jaxpr.

During tracing, arguments become **tracers** that record ops. Side effects like `append` still run during the trace, but they never appear in the compiled program. For debug output inside `jit`, use [`jax.debug.print`](https://docs.jax.dev/en/latest/_autosummary/jax.debug.print.html) instead of `print`.

In [ ]:
global_list = []


def log2(x):
    global_list.append(x)
    ln_x = jnp.log(x)
    ln_2 = jnp.log(2.0)
    return ln_x / ln_2


global_list.clear()
print(make_jaxpr(log2)(3.0))
print("global_list after trace:", global_list)

## `print` is a side effect — trace time only

`print` is not pure: it runs during tracing, but the jaxpr only captures array ops. The printed value is a tracer, not a concrete number.

In [ ]:
def log2_with_print(x):
    print("printed x:", x)
    ln_x = jnp.log(x)
    ln_2 = jnp.log(2.0)
    return ln_x / ln_2


print(make_jaxpr(log2_with_print)(3.0))

## Python control flow is traced once per branch

A jaxpr captures the function **as executed on the arguments you pass**. A Python `if` on a **static** attribute like `ndim` picks one branch; the other branch is absent from the jaxpr.

In [ ]:
def log2_if_rank_2(x):
    if x.ndim == 2:
        ln_x = jnp.log(x)
        ln_2 = jnp.log(2.0)
        return ln_x / ln_2
    return x


print("rank-1 input (else branch):")
print(make_jaxpr(log2_if_rank_2)(jnp.array([1, 2, 3])))
print("rank-2 input (if branch):")
print(make_jaxpr(log2_if_rank_2)(jnp.ones((2, 3))))

## What `jit` does

Without `jit`, JAX dispatches one op at a time — XLA cannot fuse or optimize across your function. `jit` traces into a jaxpr, lowers to **StableHLO**, compiles once per cache key (shapes/dtypes/static args), and reuses the executable. New shapes → **retrace**.

Warm up before timing (first call pays compile cost). Use `block_until_ready()` because JAX dispatch is [asynchronous](https://docs.jax.dev/en/latest/async_dispatch.html).

## SELU: eager ops vs one fused kernel

In [ ]:
def selu(x, alpha=1.67, lambda_=1.05):
    return lambda_ * jnp.where(x > 0, x, alpha * jnp.exp(x) - alpha)


x_big = jnp.arange(1_000_000)
selu_jit = jax.jit(selu)

# warm up jit cache before timing steady state
jax.block_until_ready(selu_jit(x_big))

print("eager (op-by-op):")
%timeit jax.block_until_ready(selu(x_big))

print("jit (fused):")
%timeit jax.block_until_ready(selu_jit(x_big))

## A 3-layer MLP to inspect

In [ ]:
def mlp3(params, x):
    for layer in params:
        x = jnp.tanh(x @ layer["w"] + layer["b"])
    return x


def init_mlp(key, d_in, d_hidden, d_out):
    key, k0, k1, k2 = jr.split(key, 4)
    return [
        {"w": jr.normal(k0, (d_in, d_hidden)) * 0.1, "b": jnp.zeros(d_hidden)},
        {"w": jr.normal(k1, (d_hidden, d_hidden)) * 0.1, "b": jnp.zeros(d_hidden)},
        {"w": jr.normal(k2, (d_hidden, d_out)) * 0.1, "b": jnp.zeros(d_out)},
    ]


key = jr.key(0)
params = init_mlp(key, 16, 32, 4)
x = jr.normal(jr.key(1), (8, 16))

print(make_jaxpr(mlp3)(params, x))

## `jit` wraps the jaxpr in a compiled call

In [ ]:
jitted_mlp = jax.jit(mlp3)
print(make_jaxpr(jitted_mlp)(params, x))

## First call vs steady state

In [ ]:
fresh = jax.jit(mlp3)

print("first call (compile+run):")
%timeit -n 1 -r 1 jax.block_until_ready(fresh(params, x))

for _ in range(3):
    jax.block_until_ready(fresh(params, x))

print("steady state (run only):")
%timeit jax.block_until_ready(fresh(params, x))

## Retrace on shape change

In [ ]:
compile_log = []


def mlp_trace(params, x):
    compile_log.append(x.shape)
    return mlp3(params, x)


jitted_trace = jax.jit(mlp_trace)
x_a = jr.normal(jr.key(2), (4, 16))
x_b = jr.normal(jr.key(3), (8, 16))

jitted_trace(params, x_a)
jitted_trace(params, x_a)
jitted_trace(params, x_b)
print("shapes seen at trace time:", compile_log)

## Why you can't `jit` everything

Inside `jit`, traced array **values** cannot drive Python `if` / `while` — only **static** attributes like `shape` and `dtype` can. Condition on a runtime value → `TracerBoolConversionError`.

Fixes: rewrite with array ops (`jnp.where`), use [control-flow primitives](https://docs.jax.dev/en/latest/jax.lax.html#lax-control-flow) like `jax.lax.cond`, mark value-dependent args as **static**, or `jit` only the expensive inner body.

In [ ]:
def f(x):
    if x > 0:
        return x
    return 2 * x


def g(x, n):
    i = 0
    while i < n:
        i += 1
    return x + i


for label, fn, args in [
    ("if x > 0", lambda: jax.jit(f)(jnp.array(10))),
    ("while i < n", lambda: jax.jit(g)(10, 20)),
]:
    try:
        fn()
    except jax.errors.TracerBoolConversionError as e:
        print(f"{label}: TracerBoolConversionError")

## Partial `jit` — compile the hot inner function

When Python control flow must stay outside `jit`, compile just the loop body. Define the jitted function **once** at module scope so the cache can hit.

In [ ]:
@jax.jit
def loop_body(prev_i):
    return prev_i + 1


def g_inner_jitted(x, n):
    i = 0
    while i < n:
        i = loop_body(i)
    return x + i


print(g_inner_jitted(10, 20))

## Static vs traced — what `jit` can and cannot specialize on

| Kind | Examples | At `jit` time |
|------|----------|----------------|
| **Traced** | `jnp` arrays | Shapes and dtypes are baked into the compiled program. New shapes → **retrace**. |
| **Static** | Python `int`, `bool`, `str`, `None` that control structure | Mark with `static_argnums` / `static_argnames`. Each distinct static value → **recompile**. |
| **Not compiled** | `print`, file I/O, mutation, arbitrary Python loops over Python lists | Run at **trace** time only (or not at all inside compiled code). |

**Rule of thumb:** tensor data → traced; hyperparameters and flags that pick branches → static (only when the set of values is small).

## `static_argnums` for Python scalars

In [ ]:
def repeat_matmul(x, n):
    for _ in range(n):
        x = x @ x
    return x


jitted_repeat = jax.jit(repeat_matmul, static_argnums=1)

a = jr.normal(jr.key(4), (64, 64))
print("n=2:", jitted_repeat(a, 2).shape)
print("n=3:", jitted_repeat(a, 3).shape)

## `static_argnames` and the decorator factory

Same idea as `static_argnums`, but by parameter name. With the decorator, pass static options to the factory: `@jax.jit(static_argnames=["n"])`.

In [ ]:
f_jit = jax.jit(f, static_argnums=0)
print("f_jit(10):", f_jit(10))

g_jit = jax.jit(g, static_argnames=["n"])
print("g_jit(10, 20):", g_jit(10, n=20))


@jax.jit(static_argnames=["n"])
def g_jit_decorated(x, n):
    i = 0
    while i < n:
        i += 1
    return x + i


print("g_jit_decorated(10, 20):", g_jit_decorated(10, 20))

## Print inside `jit` fires at **trace** time, not every run

In [ ]:
print("--- outside jit ---")


@jax.jit
def noisy_square(x):
    print("inside jitted function, x.shape =", x.shape)
    return x ** 2


v = jnp.ones((3,))
print("run 1:")
noisy_square(v)
print("run 2 (same shape):")
noisy_square(v)
print("run 3 (new shape):")
noisy_square(jnp.ones((5,)))

## JIT caching — don't redefine functions in loops

`jax.jit` caches by function identity. Calling `jit` on a **new** function object each iteration (`partial`, `lambda`) forces recompilation every time. Reusing the same function object is fine.

In [ ]:
def unjitted_loop_body(prev_i):
    return prev_i + 1


def g_inner_jitted_partial(x, n):
    i = 0
    while i < n:
        i = jax.jit(partial(unjitted_loop_body))(i)
    return x + i


def g_inner_jitted_lambda(x, n):
    i = 0
    while i < n:
        i = jax.jit(lambda v: unjitted_loop_body(v))(i)
    return x + i


def g_inner_jitted_normal(x, n):
    i = 0
    while i < n:
        i = jax.jit(unjitted_loop_body)(i)
    return x + i


print("partial (bad):")
%timeit jax.block_until_ready(g_inner_jitted_partial(10, 20))

print("lambda (bad):")
%timeit jax.block_until_ready(g_inner_jitted_lambda(10, 20))

print("same fn (ok):")
%timeit jax.block_until_ready(g_inner_jitted_normal(10, 20))

> **Key insight:** JIT does not execute your Python code at runtime — it traces it once. Print statements inside jitted functions fire at trace time, not run time. This confusion catches everyone.

---
## Exercise

1. `make_jaxpr` on the 3-layer MLP — find a `dot_general` or `dot` primitive.
2. Show a side effect (`append`) that runs at trace time but is absent from the jaxpr.
3. Trigger `TracerBoolConversionError` with `if x > 0` inside `jit`.
4. Fix a value-dependent loop with `static_argnums` or `static_argnames`.
5. Change batch size and log how many distinct shapes your jitted function sees.
6. Time first vs steady-state `jit` call on the same shapes (warm up first).
7. Compare `jit(partial(f))` in a loop vs `jit(f)` — which recompiles?

*(Solution below.)*

In [ ]:
print("jaxpr snippet:")
print(str(make_jaxpr(jitted_mlp)(params, x))[:400], "...")

**Next:** Episode 3 — `grad`, `value_and_grad`, and checkpointing.